# Massive Historical Underlying Access Probe — 2021 and 2022

This notebook answers a narrow question before any additional dissertation data are downloaded:

> **Does the current Massive account return one-minute SPY, SPX and VIX data for 2022 and/or 2021?**

It deliberately **does not write to the dissertation DuckDB database** and does not alter the existing raw-data folders.

A 2023 date is tested as a control because the current project expects 2023 access. The notebook then probes several ordinary trading dates in 2022 and 2021.

## Interpretation

For each ticker/year:

- `AVAILABLE` — at least one probe date returned aggregate rows;
- `NO_ROWS` — requests completed but no aggregate rows were returned;
- `ERROR_OR_ENTITLEMENT` — the API returned an exception/error;
- `MIXED` — different probe dates produced different outcomes.

Because data entitlements can differ between stocks and indices, SPY, SPX and VIX are reported separately.

If 2022 or 2021 works, we can then extend the main robustness pipeline further back without changing its methodology.

## 1. Environment and Massive client

In [4]:
from pathlib import Path
import os
import sys
import json

import pandas as pd
from dotenv import load_dotenv, dotenv_values
SRC_ROOT = PROJECT_ROOT / "src"
API_BASE = "https://api.massive.com"
PROJECT_ROOT = Path.cwd()
ENV_PATH = PROJECT_ROOT / "main.env"

print("Notebook working directory:", PROJECT_ROOT)
print("Expected environment file:", ENV_PATH)
print("File exists:", ENV_PATH.is_file())

if not ENV_PATH.is_file():
    print("Files containing 'env' in the project directory:")
    print([path.name for path in PROJECT_ROOT.glob("*env*")])
    raise FileNotFoundError(f"Environment file not found: {ENV_PATH}")

# Parse the file without putting its values into os.environ.
parsed = dotenv_values(ENV_PATH)

print("Parsed variable names:", list(parsed.keys()))
print("MASSIVE_API_KEY found:", "MASSIVE_API_KEY" in parsed)
print("MASSIVE_API_KEY has a value:", bool(parsed.get("MASSIVE_API_KEY")))

# override=True avoids a stale or empty variable in the notebook kernel
loaded = load_dotenv(ENV_PATH, override=True, verbose=True)

API_KEY = os.getenv("MASSIVE_API_KEY")

print("load_dotenv loaded variables:", loaded)
print("API key available:", bool(API_KEY))
print("API key length:", len(API_KEY) if API_KEY else 0)

if not API_KEY:
    raise RuntimeError("MASSIVE_API_KEY is missing from main.env/environment.")

sys.path.insert(0, str(SRC_ROOT))

from massive_database import MassiveREST, normalize_aggregates

client = MassiveREST(API_KEY)

OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "underlying_history_probe"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Probe output:", OUTPUT_ROOT)
print("API key loaded:", bool(API_KEY))

Notebook working directory: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation
Expected environment file: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\main.env
File exists: True
Parsed variable names: ['MASSIVE_API_KEY']
MASSIVE_API_KEY found: True
MASSIVE_API_KEY has a value: True
load_dotenv loaded variables: True
API key available: True
API key length: 32
Project root: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation
Probe output: c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\outputs\underlying_history_probe
API key loaded: True


## 2. Probe dates

In [5]:
# Multiple ordinary weekdays are used so that one unusual/missing session
# does not determine the conclusion for an entire year.
PROBE_DATES = {
    2023: [
        "2023-02-15",
        "2023-05-15",
        "2023-08-15",
        "2023-11-15",
    ],
    2022: [
        "2022-02-15",
        "2022-05-16",
        "2022-08-15",
        "2022-11-15",
    ],
    2021: [
        "2021-02-16",
        "2021-05-17",
        "2021-08-16",
        "2021-11-15",
    ],
}

TICKERS = [
    ("SPY", "stock"),
    ("I:SPX", "index"),
    ("I:VIX", "index"),
]

display(
    pd.DataFrame(
        [
            {"year": year, "probe_date": date}
            for year, dates in PROBE_DATES.items()
            for date in dates
        ]
    )
)

,year,probe_date
0,2023,2023-02-15
1,2023,2023-05-15
2,2023,2023-08-15
3,2023,2023-11-15
4,2022,2022-02-15
5,2022,2022-05-16
6,2022,2022-08-15
7,2022,2022-11-15
8,2021,2021-02-16
9,2021,2021-05-17


## 3. Run the non-destructive API probe

In [6]:
def probe_one_day(ticker: str, asset_class: str, probe_date: str) -> dict:
    result = {
        "ticker": ticker,
        "asset_class": asset_class,
        "probe_date": probe_date,
        "year": int(probe_date[:4]),
        "status": None,
        "raw_rows": 0,
        "normalised_rows": 0,
        "first_timestamp": None,
        "last_timestamp": None,
        "error": None,
    }

    try:
        raw = client.aggregates(
            ticker,
            probe_date,
            probe_date,
        )

        raw_rows = len(raw) if raw is not None else 0
        result["raw_rows"] = raw_rows

        if not raw:
            result["status"] = "NO_ROWS"
            return result

        frame = normalize_aggregates(
            raw,
            ticker,
            asset_class,
        )

        result["normalised_rows"] = len(frame)

        if len(frame):
            timestamp_col = (
                "timestamp_et"
                if "timestamp_et" in frame.columns
                else (
                    "timestamp_utc"
                    if "timestamp_utc" in frame.columns
                    else None
                )
            )

            if timestamp_col:
                result["first_timestamp"] = frame[timestamp_col].min()
                result["last_timestamp"] = frame[timestamp_col].max()

        result["status"] = "AVAILABLE"
        return result

    except Exception as exc:
        result["status"] = "ERROR_OR_ENTITLEMENT"
        result["error"] = str(exc)[:1000]
        return result


rows = []

for ticker, asset_class in TICKERS:
    for year, dates in PROBE_DATES.items():
        for probe_date in dates:
            print(f"Testing {ticker} on {probe_date}...")
            rows.append(
                probe_one_day(
                    ticker=ticker,
                    asset_class=asset_class,
                    probe_date=probe_date,
                )
            )

probe_results = pd.DataFrame(rows)

display(probe_results)

Testing SPY on 2023-02-15...
Testing SPY on 2023-05-15...
Testing SPY on 2023-08-15...
Testing SPY on 2023-11-15...
Testing SPY on 2022-02-15...
Testing SPY on 2022-05-16...
Testing SPY on 2022-08-15...
Testing SPY on 2022-11-15...
Testing SPY on 2021-02-16...
Testing SPY on 2021-05-17...
Testing SPY on 2021-08-16...
Testing SPY on 2021-11-15...
Testing I:SPX on 2023-02-15...
Testing I:SPX on 2023-05-15...
Testing I:SPX on 2023-08-15...
Testing I:SPX on 2023-11-15...
Testing I:SPX on 2022-02-15...
Testing I:SPX on 2022-05-16...
Testing I:SPX on 2022-08-15...
Testing I:SPX on 2022-11-15...
Testing I:SPX on 2021-02-16...
Testing I:SPX on 2021-05-17...
Testing I:SPX on 2021-08-16...
Testing I:SPX on 2021-11-15...
Testing I:VIX on 2023-02-15...
Testing I:VIX on 2023-05-15...
Testing I:VIX on 2023-08-15...
Testing I:VIX on 2023-11-15...
Testing I:VIX on 2022-02-15...
Testing I:VIX on 2022-05-16...
Testing I:VIX on 2022-08-15...
Testing I:VIX on 2022-11-15...
Testing I:VIX on 2021-02-16...
T

,ticker,asset_class,probe_date,year,status,raw_rows,normalised_rows,first_timestamp,last_timestamp,error
0,SPY,stock,2023-02-15,2023,AVAILABLE,824,824,2023-02-15 04:00:00,2023-02-15 19:59:00,NaN
1,SPY,stock,2023-05-15,2023,AVAILABLE,794,794,2023-05-15 04:00:00,2023-05-15 19:59:00,NaN
2,SPY,stock,2023-08-15,2023,AVAILABLE,851,851,2023-08-15 04:00:00,2023-08-15 19:59:00,NaN
3,SPY,stock,2023-11-15,2023,AVAILABLE,817,817,2023-11-15 04:00:00,2023-11-15 19:59:00,NaN
4,SPY,stock,2022-02-15,2022,AVAILABLE,887,887,2022-02-15 04:00:00,2022-02-15 19:59:00,NaN
5,SPY,stock,2022-05-16,2022,AVAILABLE,829,829,2022-05-16 04:00:00,2022-05-16 19:59:00,NaN
6,SPY,stock,2022-08-15,2022,AVAILABLE,831,831,2022-08-15 04:00:00,2022-08-15 19:59:00,NaN
7,SPY,stock,2022-11-15,2022,AVAILABLE,918,918,2022-11-15 04:00:00,2022-11-15 19:59:00,NaN
8,SPY,stock,2021-02-16,2021,ERROR_OR_ENTITLEMENT,0,0,NaT,NaT,Massive API returned HTTP 403: Your plan doesn...
9,SPY,stock,2021-05-17,2021,ERROR_OR_ENTITLEMENT,0,0,NaT,NaT,Massive API returned HTTP 403: Your plan doesn...


## 4. Summarise access by ticker and year

In [7]:
summary_rows = []

for (ticker, asset_class, year), group in probe_results.groupby(
    ["ticker", "asset_class", "year"]
):
    statuses = set(group["status"].dropna())

    if statuses == {"AVAILABLE"}:
        overall = "AVAILABLE"
    elif "AVAILABLE" in statuses:
        overall = "MIXED"
    elif statuses == {"NO_ROWS"}:
        overall = "NO_ROWS"
    elif statuses == {"ERROR_OR_ENTITLEMENT"}:
        overall = "ERROR_OR_ENTITLEMENT"
    else:
        overall = "MIXED"

    summary_rows.append(
        {
            "ticker": ticker,
            "asset_class": asset_class,
            "year": year,
            "overall_status": overall,
            "probe_dates": len(group),
            "available_dates": int(group["status"].eq("AVAILABLE").sum()),
            "no_row_dates": int(group["status"].eq("NO_ROWS").sum()),
            "error_dates": int(
                group["status"].eq("ERROR_OR_ENTITLEMENT").sum()
            ),
            "max_rows_returned": int(group["normalised_rows"].max()),
            "example_error": (
                group.loc[
                    group["error"].notna(),
                    "error",
                ].iloc[0]
                if group["error"].notna().any()
                else None
            ),
        }
    )

access_summary = (
    pd.DataFrame(summary_rows)
    .sort_values(["year", "ticker"], ascending=[False, True])
    .reset_index(drop=True)
)

display(access_summary)

,ticker,asset_class,year,overall_status,probe_dates,available_dates,no_row_dates,error_dates,max_rows_returned,example_error
0,I:SPX,index,2023,AVAILABLE,4,4,0,0,394,NaN
1,I:VIX,index,2023,AVAILABLE,4,4,0,0,735,NaN
2,SPY,stock,2023,AVAILABLE,4,4,0,0,851,NaN
3,I:SPX,index,2022,NO_ROWS,4,0,4,0,0,NaN
4,I:VIX,index,2022,NO_ROWS,4,0,4,0,0,NaN
5,SPY,stock,2022,AVAILABLE,4,4,0,0,918,NaN
6,I:SPX,index,2021,NO_ROWS,4,0,4,0,0,NaN
7,I:VIX,index,2021,NO_ROWS,4,0,4,0,0,NaN
8,SPY,stock,2021,MIXED,4,2,0,2,784,Massive API returned HTTP 403: Your plan doesn...


## 5. Simple decision table

In [8]:
decision = access_summary.pivot(
    index="ticker",
    columns="year",
    values="overall_status",
)

display(decision)

for year in [2022, 2021]:
    year_rows = access_summary[
        access_summary["year"].eq(year)
    ]

    all_available = (
        len(year_rows) == len(TICKERS)
        and year_rows["overall_status"].isin(
            ["AVAILABLE", "MIXED"]
        ).all()
    )

    if all_available:
        print(
            f"{year}: all three underlying series returned data on at least "
            "one probe date. A fuller historical retrieval is worth testing."
        )
    else:
        print(
            f"{year}: at least one required underlying did not return usable "
            "data in this probe. Review the per-ticker statuses/errors above "
            "before extending the dissertation pipeline."
        )

year,2021,2022,2023
ticker,,,
I:SPX,NO_ROWS,NO_ROWS,AVAILABLE
I:VIX,NO_ROWS,NO_ROWS,AVAILABLE
SPY,MIXED,AVAILABLE,AVAILABLE


2022: at least one required underlying did not return usable data in this probe. Review the per-ticker statuses/errors above before extending the dissertation pipeline.
2021: at least one required underlying did not return usable data in this probe. Review the per-ticker statuses/errors above before extending the dissertation pipeline.


## 6. Save the probe evidence

These files are useful for documenting exactly what the API returned on the day the historical-access check was performed.

In [9]:
detail_path = OUTPUT_ROOT / "massive_2021_2022_probe_detail.csv"
summary_path = OUTPUT_ROOT / "massive_2021_2022_probe_summary.csv"
manifest_path = OUTPUT_ROOT / "massive_2021_2022_probe_manifest.json"

probe_results.to_csv(detail_path, index=False)
access_summary.to_csv(summary_path, index=False)

manifest = {
    "purpose": "Check historical SPY/SPX/VIX minute-data access before expanding dissertation history",
    "years_tested": sorted(PROBE_DATES),
    "tickers": [ticker for ticker, _ in TICKERS],
    "probe_dates": PROBE_DATES,
    "database_modified": False,
    "raw_data_folders_modified": False,
    "interpretation_note": (
        "The probe reports observed API responses. A successful date indicates "
        "that historical aggregate data were returned for that ticker/date; "
        "it does not itself guarantee every trading day in the year is complete."
    ),
}

manifest_path.write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8",
)

print("Saved:")
print(" -", detail_path)
print(" -", summary_path)
print(" -", manifest_path)

Saved:
 - c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\outputs\underlying_history_probe\massive_2021_2022_probe_detail.csv
 - c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\outputs\underlying_history_probe\massive_2021_2022_probe_summary.csv
 - c:\Users\hkhat\OneDrive\Desktop\Project\Dissertation\outputs\underlying_history_probe\massive_2021_2022_probe_manifest.json


## 7. What to do with the result

### If 2022 and 2021 both work for SPY, SPX and VIX

Do **not** immediately mix them into the already-completed RQ5 options backtest. Instead:

1. change the extended underlying start date;
2. rebuild RQ1–RQ4 with the same feature definitions and temporal methodology;
3. describe the longer history as an underlying-market robustness extension;
4. leave the two-year options/RQ5 period unchanged because option availability is a separate constraint.

### If SPY works but one or both index series fail

The current dissertation feature set cannot be recreated consistently for that year, so do not silently substitute a different source in the same robustness analysis.

### If the API returns an entitlement error

Keep the exact error in the saved probe report. It documents that the limitation came from available historical access rather than an arbitrary analytical choice.